# FireEye — Dataset Preparation

Downloads all 4 source datasets, normalizes class IDs, merges, augments, and uploads to HuggingFace.

**Run time**: ~3–5 hours on Colab (mostly download + augmentation)

**Before running:**
1. Add `HF_TOKEN` to Colab Secrets (key icon in left sidebar)
2. Add `KAGGLE_USERNAME` and `KAGGLE_KEY` to Colab Secrets
3. Make sure Google Drive has 25+ GB free

In [1]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_ROOT = '/content/drive/MyDrive/fireye'
os.makedirs(REPO_ROOT, exist_ok=True)
os.chdir(REPO_ROOT)
print(f'Working directory: {os.getcwd()}')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# ── Cell 2: Clone repo and install deps ─────────────────────────────────────
import os

if not os.path.exists('/content/drive/MyDrive/fireye/scripts'):
    !git clone https://github.com/hajorda/fireye.git /content/drive/MyDrive/fireye
else:
    print('Repo already cloned.')

!pip install -q ultralytics albumentations datasets huggingface_hub imagehash kaggle tqdm opencv-python-headless Pillow

In [ ]:
# ── Cell 3: Set up credentials from Colab Secrets ───────────────────────────
import os
from google.colab import userdata

# HuggingFace token
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# Kaggle credentials
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Write kaggle.json so the kaggle CLI works
import json
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump({'username': os.environ['KAGGLE_USERNAME'], 'key': os.environ['KAGGLE_KEY']}, f)
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
print('Credentials configured.')

In [ ]:
# ── Cell 4: Download D-Fire (Kaggle) ────────────────────────────────────────
# ~1.2 GB download
!python scripts/download_datasets.py --root data/raw --datasets dfire

In [ ]:
# ── Cell 5: Download Pyro-SDIS (HuggingFace) ────────────────────────────────
# ~4 GB — serializes to disk as images + YOLO .txt labels
!python scripts/download_datasets.py --root data/raw --datasets pyro_sdis

In [ ]:
# ── Cell 6: Download AI for Mankind (GitHub, 50k sample) ────────────────────
# Samples 50k images to stay within Drive limits
!python scripts/download_datasets.py --root data/raw --datasets aiformankind --aiformankind-max 50000

In [ ]:
# ── Cell 7: Download Catargiu 2024 (GitHub) ──────────────────────────────────
!python scripts/download_datasets.py --root data/raw --datasets catargiu

In [ ]:
# ── Cell 8: Normalize class IDs ──────────────────────────────────────────────
# Remaps all sources to fire=0, smoke=1
!python scripts/normalize_labels.py --raw-root data/raw --processed-root data/processed

In [ ]:
# ── Cell 9: Merge + deduplicate + split ──────────────────────────────────────
# 70/20/10 stratified split; pHash deduplication (threshold=8)
!python scripts/merge_datasets.py --processed-root data/processed --merged-root data/merged

In [ ]:
# ── Cell 10: Offline augmentation (train split only) ─────────────────────────
# Generates 2× variants per image (4× for fire-only images)
# This is the slowest step — ~1–2 hours
!python scripts/augment_dataset.py --merged-root data/merged --multiplier 2 --fire-multiplier 4

In [ ]:
# ── Cell 11: Verify dataset integrity ────────────────────────────────────────
!python scripts/verify_dataset.py --merged-root data/merged

# Show the sample grid
from IPython.display import Image as IPImage
IPImage('data/merged/sample_grid.jpg', width=800)

In [ ]:
# ── Cell 12: Push to HuggingFace Hub ─────────────────────────────────────────
# Creates repo: hajorda/fireye-wildfire-detection
# ~30–60 min depending on dataset size and shard size
!python scripts/push_to_hub.py --merged-root data/merged

## Done!

Dataset is now live at: https://huggingface.co/datasets/hajorda/fireye-wildfire-detection

Open `02_train_yolov8m.ipynb` to start training.